# 6 组 Cluster Finding

对 real data 和 permutation data 分别找 cluster，保存结果。

| # | group_id | CSV 后缀 | tail | 含义 |
|---|----------|---------|------|------|
| 1 | a_tail1 | `_a.csv` | 1 | 正的 a 路径效应 |
| 2 | a_tail-1 | `_a.csv` | -1 | 负的 a 路径效应（分散学习带来更多激活） |
| 3 | ab_a_gt_0_tail1 | `_ab_a大于0.csv` | 1 | 正的间接效应（分散学习相比集中学习有更多理解加成） |
| 4 | ab_a_le_0_tail1 | `_ab_a小于等于0.csv` | 1 | 正的间接效应（a≤0 子集） |
| 5 | ab_a_gt_0_tail-1 | `_ab_a大于0.csv` | -1 | 负的间接效应（a>0 子集） |
| 6 | ab_a_le_0_tail-1 | `_ab_a小于等于0.csv` | -1 | 负的间接效应（集中学习相比分散学习有更多理解加成） |

**输出格式**（与原 utils_sumtPermu.ipynb 一致）：
- Real data: pickle 文件 `(clustersT, clusterT_sum)`
- Permutation: CSV 文件 `min_max_cluster_sums_{group_id}.csv`，含 File/MinClusterSum/MaxClusterSum 列

In [ ]:
import sys
import os

# notebook 在 utils_EEG/ 内，需要把父目录加到 sys.path 才能 import utils_EEG.stuff
sys.path.insert(0, os.path.abspath('..'))

import glob
import pickle
import mne
import mne.stats.cluster_level_backup as cluster_level_backup
from importlib import reload
import utils_EEG.stuff as stuff
from mne.channels import find_ch_adjacency
import pandas as pd
import numpy as np
from datetime import datetime

reload(stuff)

## Adjacency 初始化

In [ ]:
# 59 channels (excluding FT9, FT10, TP9, TP10)
channels_all = [
    'Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4',
    'O1', 'O2', 'F7', 'F8', 'T7', 'T8', 'P7', 'P8',
    'Fz', 'Cz', 'Pz', 'FC1', 'FC2', 'CP1', 'CP2',
    'FC5', 'FC6', 'CP5', 'CP6', 'F1', 'F2', 'C1',
    'C2', 'P1', 'P2', 'AF3', 'AF4', 'FC3', 'FC4',
    'CP3', 'CP4', 'PO3', 'PO4', 'F5', 'F6', 'C5',
    'C6', 'P5', 'P6', 'AF7', 'AF8', 'FT7', 'FT8',
    'TP7', 'TP8', 'PO7', 'PO8', 'Fpz', 'CPz', 'POz', 'Oz'
]

file_path = r"D:\LYW\pre10\data\7evoked_allWords\prex006-ave.fif"
evoked_example = mne.read_evokeds(file_path, proj=False, verbose=None)
evoked_example = evoked_example[0].resample(91)
evoked_example.pick_channels(channels_all)

info = evoked_example.info
adjacency, ch_names = find_ch_adjacency(info, ch_type="eeg")

# Cluster 参数
max_step = 1
t_power = 1
n_tests = 59 * 91  # 5369
n_times = 91
adjacency2 = cluster_level_backup._setup_adjacency(adjacency, n_tests, n_times)
test_iterations = 5733  # useless but required by function signature

print(f"adjacency2 ready, n_tests={n_tests}, n_times={n_times}")

## calculate_sumT_from_csv_file（从 utils_sumtPermu.ipynb 复制）

In [ ]:
def calculate_sumT_from_csv_file(
        csv_file_path,
        adjacency2, 
        test_iterations,
        sigma, 
        threshold_sum_t, 
        max_step, 
        t_power, 
        threshold_type,
        tail=0
):
    print(f"Processing {csv_file_path}")

    # 从 CSV 计算 t 值（含可选 smoothing）
    t_obs_orig_permu = stuff.apply_lmer_models_to_array_parallel_hat(
        array_of_dfs=None,
        test_iterations=test_iterations,
        max_workers=5,
        sigma=sigma,
        csv_file_path=csv_file_path
    )

    data = pd.read_csv(csv_file_path)
    p_values = data['p_value'].values

    print(f"t range: [{np.min(t_obs_orig_permu):.4f}, {np.max(t_obs_orig_permu):.4f}]")

    out = cluster_level_backup._find_clusters_TorP(
        x=t_obs_orig_permu,
        threshold=threshold_sum_t,
        tail=tail,
        adjacency=adjacency2,
        max_step=max_step,
        include=None,
        partitions=None,
        t_power=t_power,
        show_info=True,
        threshold_type=threshold_type,
        p_values=p_values
    )
    clustersT, clusterT_sum = out

    return csv_file_path, {'out': out}

## 6 组配置

In [ ]:
# 所有组共用的参数
COMMON_PARAMS = dict(
    threshold_sum_t=1.96,
    sigma=0.01,
    threshold_type='t',
    max_step=max_step,
    t_power=t_power,
)

# 6 组配置：csv_suffix 对应 prepare_permutation_csvs.ipynb 生成的文件后缀
GROUP_CONFIGS = [
    {'group_id': 'a_tail1',           'csv_suffix': '_a',             'tail': 1},
    {'group_id': 'a_tail-1',          'csv_suffix': '_a',             'tail': -1},
    {'group_id': 'ab_a_gt_0_tail1',   'csv_suffix': '_ab_a大于0',     'tail': 1},
    {'group_id': 'ab_a_le_0_tail1',   'csv_suffix': '_ab_a小于等于0',  'tail': 1},
    {'group_id': 'ab_a_gt_0_tail-1',  'csv_suffix': '_ab_a大于0',     'tail': -1},
    {'group_id': 'ab_a_le_0_tail-1',  'csv_suffix': '_ab_a小于等于0',  'tail': -1},
]

# 路径配置
RESULTS_DIR = r"D:\LYW\R for permutation t\Mplus\results"
RAW_CSV_NAME = "all_spatiotemporal_results_WLSMV_precise_parallel"
OUTPUT_DIR = r"D:\LYW\pre10\data\permutation_summary_between"

def get_csv_path(base_dir, csv_suffix):
    """根据 base_dir 和 csv_suffix 构建完整 CSV 路径。"""
    return os.path.join(base_dir, f"{RAW_CSV_NAME}{csv_suffix}.csv")

print(f"Results dir: {RESULTS_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Groups: {len(GROUP_CONFIGS)}")
for g in GROUP_CONFIGS:
    csv_path = get_csv_path(RESULTS_DIR, g['csv_suffix'])
    exists = os.path.exists(csv_path)
    print(f"  {g['group_id']:25s} tail={g['tail']:+d}  CSV exists: {exists}")

## Real data: 找 cluster 并保存 pickle

In [ ]:
current_date = datetime.now().strftime("%Y%m%d")

for i, cfg in enumerate(GROUP_CONFIGS):
    group_id = cfg['group_id']
    csv_path = get_csv_path(RESULTS_DIR, cfg['csv_suffix'])
    tail = cfg['tail']

    print(f"\n[{i+1}/{len(GROUP_CONFIGS)}] {group_id} (tail={tail})")
    print(f"  CSV: {os.path.basename(csv_path)}")

    _, result = calculate_sumT_from_csv_file(
        csv_path, adjacency2, test_iterations,
        COMMON_PARAMS['sigma'], COMMON_PARAMS['threshold_sum_t'],
        COMMON_PARAMS['max_step'], COMMON_PARAMS['t_power'],
        COMMON_PARAMS['threshold_type'], tail=tail
    )

    clustersT, clusterT_sum = result['out']
    print(f"  Found {len(clustersT)} clusters, sum range: ", end="")
    if len(clusterT_sum) > 0:
        print(f"[{clusterT_sum.min():.2f}, {clusterT_sum.max():.2f}]")
    else:
        print("(no clusters)")

    # 保存 pickle，格式与原 notebook 一致
    pickle_name = f"clusters_{current_date}_thresh_{COMMON_PARAMS['threshold_sum_t']}_direction_{tail}_{RAW_CSV_NAME}{cfg['csv_suffix']}.pkl"
    pickle_path = os.path.join(OUTPUT_DIR, pickle_name)
    with open(pickle_path, 'wb') as f:
        pickle.dump((clustersT, clusterT_sum), f)
    print(f"  Saved: {pickle_name}")

## Permutation: 找 cluster 并保存 min/max sums

In [ ]:
permu_dirs = sorted(glob.glob(os.path.join(RESULTS_DIR, "permutation_[0-9]*")))
print(f"Found {len(permu_dirs)} permutation directories")

for cfg in GROUP_CONFIGS:
    group_id = cfg['group_id']
    tail = cfg['tail']
    print(f"\n{'='*60}")
    print(f"Group: {group_id} (tail={tail})")
    print(f"{'='*60}")

    cluster_sums_stats = []

    for j, permu_dir in enumerate(permu_dirs):
        csv_path = get_csv_path(permu_dir, cfg['csv_suffix'])
        if not os.path.exists(csv_path):
            print(f"  SKIP {os.path.basename(permu_dir)}: {os.path.basename(csv_path)} not found")
            continue

        print(f"  [{j+1}/{len(permu_dirs)}] {os.path.basename(permu_dir)}")

        _, result = calculate_sumT_from_csv_file(
            csv_path, adjacency2, test_iterations,
            COMMON_PARAMS['sigma'], COMMON_PARAMS['threshold_sum_t'],
            COMMON_PARAMS['max_step'], COMMON_PARAMS['t_power'],
            COMMON_PARAMS['threshold_type'], tail=tail
        )

        clustersT, clusterT_sum = result['out']

        if len(clusterT_sum) > 0:
            min_sum = clusterT_sum.min()
            max_sum = clusterT_sum.max()
        else:
            min_sum = None
            max_sum = None

        cluster_sums_stats.append({
            'File': csv_path,
            'MinClusterSum': min_sum,
            'MaxClusterSum': max_sum
        })

    # 保存该组的 permutation 结果
    if cluster_sums_stats:
        df_stats = pd.DataFrame(cluster_sums_stats)
        output_csv = os.path.join(
            OUTPUT_DIR,
            f"min_max_cluster_sums_{current_date}_thresh_{COMMON_PARAMS['threshold_sum_t']}_{group_id}.csv"
        )
        df_stats.to_csv(output_csv, index=False)
        print(f"  Saved: {os.path.basename(output_csv)} ({len(df_stats)} permutations)")